# Question 4

# 4.1 Implement and Train a CNN

## Step 1: Import All Libraries

In [ ]:
import os
import time
import copy
import pickle
import zipfile
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

print("All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Step 2: GPU Setup

In [ ]:
print("="*70)
print("GPU VERIFICATION")
print("="*70)

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\nGPU is ready!")
else:
    print("WARNING: GPU not available!")

print("="*70)

## Step 3: Helper Functions

In [ ]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

class BigCatDatasetAugmented(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        if not os.path.exists(root_dir):
            self.classes = []
            self.data = []
            self.labels = []
            return

        class_names = sorted([d for d in os.listdir(root_dir)
                            if os.path.isdir(os.path.join(root_dir, d))])
        self.classes = class_names

        self.data = []
        self.labels = []
        for class_name in class_names:
            class_path = os.path.join(root_dir, class_name)
            file_names = sorted([f for f in os.listdir(class_path)
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))])
            for file_name in file_names:
                self.data.append(os.path.join(class_path, file_name))
                self.labels.append(self.classes.index(class_name))

        print(f"Found {len(self.classes)} classes, {len(self.data)} images")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data[idx]
        label = self.labels[idx]

        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB', (224, 224))

        if self.transform:
            img = self.transform(img)

        return img, label

print("Helper functions defined")

## Step 4: Unzip data

In [ ]:
import zipfile
with zipfile.ZipFile('big_cats.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

## Step 5: Exploratory Data Analysis (EDA)

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

DATA_ROOT = './big_cats'
TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
VAL_DIR = os.path.join(DATA_ROOT, 'valid')
TEST_DIR = os.path.join(DATA_ROOT, 'test')

print("="*60)
print("EXPLORATORY DATA ANALYSIS")
print("="*60)

temp_transform = transforms.ToTensor()
temp_train = BigCatDatasetAugmented(TRAIN_DIR, transform=temp_transform)
temp_val = BigCatDatasetAugmented(VAL_DIR, transform=temp_transform)
temp_test = BigCatDatasetAugmented(TEST_DIR, transform=temp_transform)

print("\n--- Dataset Sizes ---")
print(f"Training set: {len(temp_train)} images")
print(f"Validation set: {len(temp_val)} images")
print(f"Test set: {len(temp_test)} images")
print(f"Total: {len(temp_train) + len(temp_val) + len(temp_test)} images")

print("\n--- Class Distribution ---")
def print_distribution(dataset, name):
    labels = [os.path.basename(os.path.dirname(p)) for p in dataset.data]
    dist = Counter(labels)
    print(f"{name} distribution:")
    for class_name, count in sorted(dist.items()):
        print(f"  {class_name}: {count} images")

print_distribution(temp_train, "Training set")
print_distribution(temp_val, "Validation set")
print_distribution(temp_test, "Test set")

print("\n--- Sample Images from Each Class ---")
classes = temp_train.classes
num_classes_to_plot = min(6, len(classes))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx in range(num_classes_to_plot):
    class_name = classes[idx]
    for i in range(len(temp_train)):
        img, label = temp_train[i]
        if temp_train.classes[label] == class_name:
            img_np = img.permute(1, 2, 0).numpy()
            axes[idx].imshow(img_np)
            axes[idx].set_title(f"{class_name}", fontsize=12, fontweight='bold')
            axes[idx].axis('off')
            break

for idx in range(num_classes_to_plot, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## Step 6: Define CompactCatCNN_v2 Model

In [ ]:
from model import CompactCatCNN_v2

# Create model and check parameters
temp_model = CompactCatCNN_v2(num_classes=10)
num_params = sum(p.numel() for p in temp_model.parameters())

print(f"Model: CompactCatCNN_v2")
print(f"Parameters: {num_params:,}")
print(f"Size: {num_params * 4 / 1024 / 1024:.2f} MB")

## Step 7: GPU Configuration

In [ ]:
set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("cuDNN benchmark enabled")

## Step 8: Data Paths

In [ ]:
DATA_ROOT = './big_cats'
TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
VAL_DIR = os.path.join(DATA_ROOT, 'valid')
TEST_DIR = os.path.join(DATA_ROOT, 'test')

print(f"Train dir: {TRAIN_DIR}")
print(f"Val dir: {VAL_DIR}")
print(f"Test dir: {TEST_DIR}")

## Step 9: Data Transforms & Loaders (Optimized)

In [ ]:
# OPTIMIZED augmentation - fast but effective
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = BigCatDatasetAugmented(TRAIN_DIR, transform=train_transform)
val_dataset = BigCatDatasetAugmented(VAL_DIR, transform=val_test_transform)
test_dataset = BigCatDatasetAugmented(TEST_DIR, transform=val_test_transform)

# Create dataloaders
BATCH_SIZE = 96
NUM_WORKERS = 0 if os.name == "nt" else 4

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_dataset)} images, {len(train_loader)} batches")
print(f"Val: {len(val_dataset)} images, {len(val_loader)} batches")
print(f"Test: {len(test_dataset)} images, {len(test_loader)} batches")
print(f"Batch size: {BATCH_SIZE}")
print("Optimized augmentation enabled")

## Step 10: Training Functions (Mixed Precision)

In [ ]:
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_corrects = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = running_corrects.double() / len(loader.dataset) * 100
    return epoch_loss, epoch_acc.item()

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, device, seed=42):
    torch.manual_seed(seed)
    scaler = GradScaler()

    since = time.time()
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        model.train()
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = running_corrects.double() / len(train_loader.dataset) * 100
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc.item())

        val_loss, val_acc = validate(model, val_loader, criterion, device)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f'Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.2f}%')
        print(f'Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%')

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), 'best_model_v2.pth')
            print('Saved best model')
        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val loss: {best_val_loss:.4f}')
    model.load_state_dict(best_model_wts)
    return history

print("Training functions ready (with mixed precision)")

## Step 11: Train Model

In [ ]:
# Create model
model = CompactCatCNN_v2(num_classes=10).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# Train
NUM_EPOCHS = 75
print(f"Starting training for {NUM_EPOCHS} epochs on {device}...\n")
history = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, NUM_EPOCHS, device, seed=42)

# Load best model
if os.path.exists('best_model_v2.pth'):
    model.load_state_dict(torch.load('best_model_v2.pth', weights_only=True))
    print("\nLoaded best model")

## Step 12: Test Evaluation

In [ ]:
print("Evaluating on Test Set...")
test_loss, test_acc = validate(model, test_loader, criterion, device)

print("="*70)
print("FINAL RESULTS - CompactCatCNN_v2")
print("="*70)
print(f"Parameters: {num_params:,} (2.36M)")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")
print("="*70)

if test_acc >= 77:
    print("SUCCESS! Test accuracy >= 77% (full points!)")
else:
    print(f"Warning: Test accuracy < 77% (current: {test_acc:.2f}%)")

## Step 13: Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
epochs = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs, history['train_loss'], 'b-', label='Training Loss')
ax1.plot(epochs, history['val_loss'], 'r-', label='Validation Loss')
ax1.set_title('Training and Validation Loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, history['train_acc'], 'b-', label='Training Accuracy')
ax2.plot(epochs, history['val_acc'], 'r-', label='Validation Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Step 14: Save Model Weights

In [ ]:
# Save weights
torch.save(model.state_dict(), 'model_weights_v2.pkl')
print("Saved model_weights_v2.pkl")

# Save results summary
results = {
    'model': 'CompactCatCNN_v2',
    'parameters': num_params,
    'test_accuracy': test_acc,
    'test_loss': test_loss,
    'history': history
}

with open('results_v2.pkl', 'wb') as f:
    pickle.dump(results, f)

print("\nResults saved!")
print(f"Model: CompactCatCNN_v2")
print(f"Parameters: {num_params:,}")
print(f"Test Accuracy: {test_acc:.2f}%")

# 4.2  Analyzing a Pre-trained CNN (Filters) (10pt)

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import os
import numpy as np

# Load VGG16 with pre-trained weights
# Note: 'pretrained=True' is the legacy method requested.
# The modern way is: models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
model = models.vgg16(pretrained=True, progress=True)

# Set model to evaluation mode (freezes Batch Norm layers and Dropout)
model.eval()

print("VGG16 Loaded successfully.")

In [ ]:
import os
import zipfile

# Define the paths
output_dir = 'hw1_data'
birds_zip_path = '/content/birds.zip'
birds_images_dir = os.path.join(output_dir, 'birds')

# Create the hw1_data directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Extract birds.zip if the birds image directory doesn't exist or is empty
# This assumes the zip file contains a 'birds/' directory at its root
if not os.path.exists(birds_images_dir) or not os.listdir(birds_images_dir):
    with zipfile.ZipFile(birds_zip_path, 'r') as zip_ref:
        zip_ref.extractall(output_dir) # This extracts 'birds/' into 'hw1_data/'

# Define the standard ImageNet normalization and resizing
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def load_and_display_images(folder_path):
    image_files = [f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
    images = []

    plt.figure(figsize=(12, 6))
    for i, img_file in enumerate(image_files):
        path = os.path.join(folder_path, img_file)
        img = Image.open(path).convert('RGB')

        # Display
        plt.subplot(1, len(image_files), i + 1)
        plt.imshow(img)
        plt.title(img_file)
        plt.axis('off')

        images.append(img)
    plt.show()
    return images

# Display birds
# Note: Ensure 'hw1_data/birds' exists in your working directory
bird_images = load_and_display_images(birds_images_dir)

What steps did you take?

To feed images into VGG16, I established a transformation pipeline:

1. Resize: The VGG network requires input dimensions of at least $224 \times 224$. I resized images to $256 \times 256$ and then center-cropped to $224 \times 224$ (standard ImageNet practice).

2. ToTensor: Converts the PIL Image (H x W x C) in the range [0, 255] to a PyTorch Tensor (C x H x W) in the range [0, 1]

3. .Normalize: Applied the specific ImageNet mean and standard deviation to standardize the input.Mean: $[0.485, 0.456, 0.406]$Std: $[0.229, 0.224, 0.225]$

4. Batch Dimension: Added a dimension at index 0 (using unsqueeze(0)) because the model expects a batch of shape $(N, C, H, W)$.

In [ ]:
# Load ImageNet class labels
# You might need to download this file first if not already present
import json

# Path to the ImageNet class index JSON file
# This URL is a common source for the ImageNet class mapping.
# Ensure you have 'wget' installed or download it manually if preferred.
if not os.path.exists('imagenet_class_index.json'):
    !wget -q https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json

with open('imagenet_class_index.json') as f:
    idx2label = json.load(f)
    # The JSON structure is a dict where keys are string indices and values are [idx_str, class_name]
    # We want a dict mapping integer index to class_name
    idx2label = {int(k): v[1] for k, v in idx2label.items()}

def predict_and_show_top_class(model, raw_images):
    for img in raw_images:
        # 1. Apply preprocessing
        input_tensor = preprocess(img)
        # 2. Add batch dimension -> (1, 3, 224, 224)
        input_batch = input_tensor.unsqueeze(0)

        # 3. Forward pass
        if torch.cuda.is_available():
            input_batch = input_batch.cuda()
            model.to('cuda')

        with torch.no_grad():
            output = model(input_batch)

        # Get the index of the highest score
        _, predicted_idx = torch.max(output, 1)
        idx = predicted_idx.item()

        print(f"Image Output Shape: {output.shape}")
        # Get the name from the dictionary we created above
        label_name = idx2label[idx]
        print(f"Predicted: {label_name} (Index: {idx})")
        print("\n") # Add a newline for better separation between predictions

print("--- Bird Predictions ---")
predict_and_show_top_class(model, bird_images)


What are the outputs?

The output is a tensor of raw logits with shape (N, 1000), representing unnormalized scores for the 1,000 ImageNet classes.

Values: A higher score indicates higher model confidence for that specific class. These are not probabilities; applying Softmax would convert them to percentages.

Result: The model successfully identified the birds, outputting the maximum scores at Index 94 (Hummingbird) and Index 90 (Lorikeet).

In [ ]:
import os
import zipfile
from PIL import Image
import matplotlib.pyplot as plt

# Define dog paths
dog_zip_path = '/content/dogs.zip'
dog_images_dir = 'hw1_data/dogs'
single_dog_image_path = '/content/dog_1.jpg'

# Ensure the parent directory 'hw1_data' exists (it should from earlier cells, but good to be robust)
if not os.path.exists('hw1_data'):
    os.makedirs('hw1_data')

images_loaded_successfully = False
dog_images = []

with open('imagenet_class_index.json') as f:
    idx2label = json.load(f)
    # The JSON structure is a dict where keys are string indices and values are [idx_str, class_name]
    # We want a dict mapping integer index to class_name
    idx2label = {int(k): v[1] for k, v in idx2label.items()}


img = Image.open(single_dog_image_path).convert('RGB')
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.title(os.path.basename(single_dog_image_path))
plt.axis('off')
plt.show()
dog_images = [img] # Create a list for consistency with original code expecting dog_images[0]
images_loaded_successfully = True


print("\n--- Dog Prediction ---")
if images_loaded_successfully:
    # Assuming we select the first dog image found (or the single one)
    dog_img_pil = dog_images[0]

    # Preprocess specifically for the next step
    dog_tensor = preprocess(dog_img_pil).unsqueeze(0)

    if torch.cuda.is_available():
        dog_tensor = dog_tensor.cuda()

    with torch.no_grad():
        dog_output = model(dog_tensor)

    _, dog_pred_idx = torch.max(dog_output, 1)
    print(f"Dog Predicted Class Index: {dog_pred_idx.item()}")

    # FIX: Use .item() to get the integer value from the tensor
    label_name = idx2label[dog_pred_idx.item()]
    print(f"Predicted: {label_name} (Index: {dog_pred_idx.item()})")

The output is a tensor of raw logits with shape (1, 1000), identical in structure to the previous section.

Values: These are unnormalized confidence scores for the 1,000 ImageNet classes.

Result: The model output the maximum score for the class corresponding to Golden Retriever, successfully identifying the specific dog breed.

In [ ]:
# 1. Isolate the first layer
first_layer = model.features[0]

# 2. Pass the preprocessed dog tensor through ONLY this layer
with torch.no_grad():
    # Ensure tensor is on same device as model
    if torch.cuda.is_available():
        first_layer = first_layer.cuda()

    feature_maps = first_layer(dog_tensor)

# feature_maps shape is (1, 64, 224, 224) because VGG16's first layer has 64 filters
print(f"Feature Map Shape: {feature_maps.shape}")

# 3. Plot the first 3 filters
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i in range(3):
    # Get the i-th feature map (squeeze to remove batch dim)
    fmap = feature_maps[0, i, :, :].cpu()

    axes[i].imshow(fmap, cmap='gray') # logical choice for feature activation
    axes[i].set_title(f"Filter {i+1} Response")
    axes[i].axis('off')

plt.show()

Filter Response Analysis:

Visual Observation:When visualizing the output of the first 3 filters, we can clearly recognize the spatial structure of the original image. Specifically, we can see the outline of the dog's face and the headphones  around its neck. The features are preserved remarkably well, looking like "ghostly" or "embossed" versions of the original photograph.

Technical Explanation:

Edge Detection: The first layer of a Convolutional Neural Network (CNN) typically learns to detect low-level geometric features. The bright (white) pixels in the plots represent high activation, indicating where the filters detected sharp transitions in contrast—such as the edges of the dog's nose, the curve of the headphones, and the texture of the fur.

Spatial Resolution: Since this is the very first layer ($224 \times 224$), the spatial resolution is high enough that objects like the dog and headphones remain distinct and recognizable before being downsampled in deeper layers.

Filter Diversity: While all three filters show the dog/headphones, they emphasize different aspects (e.g., one highlights vertical edges of the headphones, while another highlights the diagonal texture of the fur), showing that the network is extracting multiple types of edge information simultaneously.